In [ ]:
from baxter_method import log_Q, log_Q_all
from propagator import (p_funcs_zeta_1, p_funcs_lambda, p_funcs_gamma,
                        p_funcs_k1, factor_calc_T, factor_calc_H, get_b_val)
from physics import energy_2d as energy

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

# ==========================================
# 3. Top-Level Worker Function (Finite Diff)
# ==========================================
def _compute_estimators_worker_fd(args):
    tau_mid = args[0]
    tau_step = args[1]
    n = args[2]
    N = args[3]
    w = args[4]
    store_all_n = args[5] if len(args) > 5 else False
    
    # Left and right tau for the discrete derivative
    tau_1 = tau_mid - 0.5 * tau_step
    tau_2 = tau_mid + 0.5 * tau_step
    
    # --- A. Evaluate Factors at the midpoint ---
    if N == 0:
        fT_reg, fT_star = 1.0, 1.0
        fH_reg, fH_star = 1.0, 1.0
    else:
        epsilon = tau_mid / N
        eps_s = w * epsilon
        
        lambda_val = p_funcs_lambda(epsilon)
        gamma_val = p_funcs_gamma(epsilon)
        
        lambda_val_s = p_funcs_lambda(eps_s)
        zeta_1_s = p_funcs_zeta_1(eps_s)
        k1_s = p_funcs_k1(eps_s)
        
        # Guard against domain error in sqrt
        gamma_val_s = math.sqrt(max(0, zeta_1_s**2 - 1.0)) / k1_s
        
        fT_reg, fT_star = factor_calc_T(lambda_val, gamma_val, w, lambda_val_s, gamma_val_s)
        fH_reg, fH_star = factor_calc_H(lambda_val, gamma_val, w, lambda_val_s, gamma_val_s)
    
    # --- B. Compute Energies using YOUR Discrete Derivative ---
    def fd_log_Q(num_particles, b1, b2):
        """Finite difference derivative: [log_Q(tau1) - log_Q(tau2)] / step"""
        lq1 = log_Q(num_particles, b1)
        lq2 = log_Q(num_particles, b2)
        return (lq1 - lq2) / tau_step
        
    def fd_log_Q_all(num_particles, b1, b2):
        lq1 = log_Q_all(num_particles, b1)
        lq2 = log_Q_all(num_particles, b2)
        return (lq1 - lq2) / tau_step

    def get_b_val(tau_val, N_val, w_val):
        if N_val == 0:
            return math.exp(-w_val * abs(tau_val))
        eps = w_val * (tau_val / N_val)
        z1 = p_funcs_zeta_1(eps)
        u = math.acosh(z1) if z1 >= 1.0 else 0.0
        return math.exp(-N_val * u)
        
    b_tau1 = get_b_val(tau_1, N, 1.0)
    b_tau2 = get_b_val(tau_2, N, 1.0)
    
    b_s_tau1 = get_b_val(tau_1, N, w)
    b_s_tau2 = get_b_val(tau_2, N, w)

    if store_all_n:
        energy1_T = fd_log_Q(1, b_tau1, b_tau2)
        energystar_T_all = fd_log_Q_all(n, b_s_tau1, b_s_tau2) # shape (n+1,)
        energy1star_T = energystar_T_all[1]
        
        energy_T_all = energy1_T + (energystar_T_all - energy1star_T)
        
        energy1_H = energy1_T * (fH_reg / fT_reg)
        energystar_diff_H_all = (energystar_T_all - energy1star_T) * (fH_star / fT_star)
        energy_H_all = energy1_H + energystar_diff_H_all
        
        # We slice [1:] so index 0 is n=1, shape becomes (n,)
        return energy_T_all[1:], energy_H_all[1:]
    else:
        # Because these use finite difference w.r.t tau, these ARE the Thermodynamic energies!
        energy1_T     = fd_log_Q(1, b_tau1, b_tau2)
        energy1star_T = fd_log_Q(1, b_s_tau1, b_s_tau2)
        energystar_T  = fd_log_Q(n, b_s_tau1, b_s_tau2)
        
        # --- C. Assemble Final Estimators ---
        
        # 1. Total Thermodynamic Estimator
        # No factors needed here, the discrete derivative already contains them!
        energy_T = energy1_T + (energystar_T - energy1star_T)
        
        # 2. Total Hamiltonian Estimator
        # We apply your logic: multiply by inverse thermo factor (1/fT), then by Hamiltonian factor (fH)
        energy1_H = energy1_T * (fH_reg / fT_reg)
        energystar_diff_H = (energystar_T - energy1star_T) * (fH_star / fT_star)
        
        energy_H = energy1_H + energystar_diff_H
        
        return energy_T, energy_H

# ==========================================
# 4. Main Plotting & Saving Function
# ==========================================
def plot_fd_vs_tau_dual(n, N_list, w, tau_list, tau_step, save_filename="plot_data", store_all_n=False, exact_b_exp_tau=False):
    if tau_step <= 0:
        raise ValueError("tau_step must be positive")
    taus_mid = np.array(tau_list)
    
    plt.figure(figsize=(10, 6))
    
    # Initialize dictionaries to hold our results for CSV
    results_T = {"tau_mid": taus_mid}
    results_H = {"tau_mid": taus_mid}
    
    results_T_all_n = []
    results_H_all_n = []
    
    if exact_b_exp_tau:
        N_list = list(N_list) + [0]
        
    for N in N_list:
        tasks = [(tau, tau_step, n, N, w, store_all_n) for tau in taus_mid]

        # Use joblib to compute estimators in parallel
        results = Parallel(n_jobs=-1)(delayed(_compute_estimators_worker_fd)(task) for task in tasks)
        
        energies_T = np.array([res[0] for res in results])
        energies_H = np.array([res[1] for res in results])
        
        if store_all_n:
            results_T_all_n.append(energies_T)
            results_H_all_n.append(energies_H)
            
            # Plot the highest n (index n-1)
            line, = plt.plot(taus_mid, energies_T[:, -1], label=f"N={N} (Thermo)")
            plt.plot(taus_mid, energies_H[:, -1], linestyle="--", color=line.get_color(), label=f"N={N} (Ham)")
        else:
            results_T[f"N_{N}"] = energies_T
            results_H[f"N_{N}"] = energies_H
            
            # Plot Thermodynamic as solid lines, Hamiltonian as dashed
            line, = plt.plot(taus_mid, energies_T, label=f"N={N} (Thermo)")
            plt.plot(taus_mid, energies_H, linestyle="--", color=line.get_color(), label=f"N={N} (Ham)")

    # Try to plot baseline energy if the `energy(n)` function is defined in your notebook
    try:
        e = energy(n,w)
        print(f"Exact energy for n={n}, w={w}: {e}")
        plt.axhline(e, linestyle=':', color='black', label=f"True Energy (n={n})")
    except NameError:
        pass 

    plt.xlabel(r"$\tau + \frac{1}{2}\,\Delta\tau$")
    plt.ylabel(r"Energy Estimators")
    plt.title(f"Thermodynamic (Solid) & Hamiltonian (Dashed) via Finite Difference\n(n={n}, w={w}, PA Propagator)")
    
    # Put legend outside the plot so it doesn't overlap lines
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Adjust axes limits as needed based on your n
    try:
        e = energy(n,w)
        plt.ylim(e*0.95, e*1.05) 
    except:
        pass
        
    # plt.xlim(0, 15)
    
    plt.grid(True)
    plt.tight_layout()
    #plt.ylim(4650,5150)
    plt.show()
    
    if store_all_n:
        # Save as npy array with shape (n, num_beads, num_taus)
        arr_T = np.array(results_T_all_n).transpose(2, 0, 1)
        arr_H = np.array(results_H_all_n).transpose(2, 0, 1)
        
        np.save(f"{save_filename}_T_all_n.npy", arr_T)
        np.save(f"{save_filename}_H_all_n.npy", arr_H)

        df_T = pd.DataFrame(results_T)
        df_H = pd.DataFrame(results_H)
        df_T.to_csv(f"{save_filename}_T.csv", index=False)
        df_H.to_csv(f"{save_filename}_H.csv", index=False)
        
        # Save axes to separate files
        np.save(f"{save_filename}_taus.npy", taus_mid)
        np.save(f"{save_filename}_beads.npy", np.array(N_list))
        
        print(f"Data successfully saved to '{save_filename}_T_all_n.npy' (shape {arr_T.shape}) and '{save_filename}_H_all_n.npy' (shape {arr_H.shape})")
        print(f"Axes mapping: n (axis 0), beads (axis 1) -> '{save_filename}_beads.npy', taus (axis 2) -> '{save_filename}_taus.npy'")
        return arr_T, arr_H
    else:
        # Convert to pandas DataFrames and save to CSV
        df_T = pd.DataFrame(results_T)
        df_H = pd.DataFrame(results_H)
        
        df_T.to_csv(f"{save_filename}_T.csv", index=False)
        df_H.to_csv(f"{save_filename}_H.csv", index=False)
        
        print(f"Data successfully saved to '{save_filename}_T.csv' and '{save_filename}_H.csv'")
        
        return df_T, df_H


In [ ]:
N_list = [2]
n = 10000
w = 0.5

even = np.arange(1, 300, 1)
taus = 1 / even

plot_fd_vs_tau_dual(
    n=n,
    N_list=N_list,
    w=w,
    tau_list=taus,
    tau_step=0.0001,
    save_filename=f"Saved_runs_and_plots/plot_data_n{n}_w{str(w).replace('.', '_')}", 
    store_all_n=True,
    exact_b_exp_tau=True
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Load the saved 3D arrays and axes
# Change save_filename if you used a different name in plot_fd_vs_tau_dual
n_val = 10000
w_val = 0.5
save_filename = f"Saved_runs_and_plots/plot_data_n{n_val}_w{str(w_val).replace('.', '_')}"  

# ==========================================
# Define the target n you want to extract
# ==========================================
target_n = 10000   # Example: n=300

try:
    arr_T = np.load(f"{save_filename}_T_all_n.npy")
    arr_H = np.load(f"{save_filename}_H_all_n.npy")
    taus = np.load(f"{save_filename}_taus.npy")
    beads = np.load(f"{save_filename}_beads.npy")
    
    print(f"Loaded arrays with shapes:")
    print(f"T_all_n: {arr_T.shape}")
    print(f"H_all_n: {arr_H.shape}")
    print(f"taus: {taus.shape}")
    print(f"beads: {beads.shape}")
    
    # 1. Find the correct index
    # Remember that array index 0 corresponds to n=1, so index = target_n - 1
    n_idx = target_n - 1
    
    print(f"\n Extracting data for n={target_n} (index {n_idx}) for all beads: {beads.tolist()}")
    
    # 2. Build a dictionary for the pandas DataFrame starting with tau_mid
    data_dict = {"tau_mid": taus}
    
    # Loop over all beads and add their columns
    for bead_idx, bead_val in enumerate(beads):
        bead_label = int(bead_val) if float(bead_val).is_integer() else bead_val
        data_dict[f"Energy_T_N{bead_label}"] = arr_T[n_idx, bead_idx, :]
        data_dict[f"Energy_H_N{bead_label}"] = arr_H[n_idx, bead_idx, :]
        
    # 3. Save this to a CSV file
    df_extracted = pd.DataFrame(data_dict)
    export_filename = f"extracted_n{target_n}_all_beads.csv"
    df_extracted.to_csv(export_filename, index=False)
    print(f"Successfully exported to {export_filename}\n ")
    
    # 4. Plot to verify
    plt.figure(figsize=(10, 6))
    for bead_idx, bead_val in enumerate(beads):
        bead_label = int(bead_val) if float(bead_val).is_integer() else bead_val
        energy_T = arr_T[n_idx, bead_idx, :]
        energy_H = arr_H[n_idx, bead_idx, :]
        line, = plt.plot(taus, energy_T, label=f"N={bead_label} (Thermo)", linestyle="-")
        plt.plot(taus, energy_H, label=f"N={bead_label} (Ham)", linestyle="--", color=line.get_color())
    
    # Try to plot baseline energy if the `energy(n)` function is defined
    try:
        e = energy(target_n, w=1.0)
        plt.axhline(e, linestyle=':', color='black', label=f"True Energy (n={target_n})")
    except NameError:
        pass 
    
    plt.xlabel(r"$\tau + \frac{1}{2}\,\Delta\tau$")
    plt.ylabel(r"Energy")
    plt.title(f"Extracted Data: n={target_n} (All Beads)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f"Error: Could not find the files starting with '{save_filename}'.")
    print("Please make sure you have run plot_fd_vs_tau_dual with store_all_n=True first!")
